# EE 5830 Final Project

## 5G-NR Signal SNR Classification Using Deep Learning

### Author: Marvin Mendez
### Date: December 11, 2024

In [88]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import json
import torch
import torch.nn as nn
import torch.nn.functional as F
import skorch
from sklearn.model_selection import train_test_split

### Loading the given parameters of the data

In [89]:
with open('RF_signal_dataset/IEEE_Dataport/5G-NR/32274CF-dell-latitude-D20211201T191100M024240.json') as p:
    parameters = json.load(p)

df_parameters = pd.json_normalize(parameters)

print(df_parameters)

   frequency  gain  sampling_rate  bitdepth  impedance
0  628000000    50       20000000        16         50


In [90]:
params = {
    "frequency": 628000000,
    "gain": 50,
    "sampling_rate": 20000000,
    "bitdepth": 16,
    "impedance": 50
}

### Loading the Data

In [91]:
# Load and preprocess I/Q data
data = np.fromfile(r"C:\Users\mende\OneDrive\Documents\ee5830\Final_Project\RF_signal_dataset\IEEE_Dataport\5G-NR\32274CF-dell-latitude-D20211201T191100M024240.data", np.int16)

data_5g_iq = data[0::2] + 1j * data[1::2]
signal_real = np.real(data_5g_iq) / (2**(params["bitdepth"] - 1))
signal_imaginary = np.imag(data_5g_iq) / (2**(params["bitdepth"] - 1))

# Normalize the data
# Normalize while preserving I/Q relationship
magnitude = np.sqrt(signal_real**2 + signal_imaginary**2)
max_magnitude = np.max(magnitude)
signal_real = signal_real / max_magnitude
signal_imaginary = signal_imaginary / max_magnitude

# Stack the real and imaginary parts to create a two-channel input
data_combined = np.stack((signal_real, signal_imaginary), axis=-1)

# Reshape the data to fit PyTorch input requirements
data_combined = data_combined.reshape(10000, 2, 2000) # Reshape to (num_samples, channels, length)

In [92]:
print(data_combined.shape) # printing shape to verify

(10000, 2, 2000)


### Importing Functions

In [93]:
import importlib
import Final_Project_funcs as fpf
importlib.reload(fpf)

<module 'Final_Project_funcs' from 'C:\\Users\\mende\\OneDrive\\Documents\\ee5830\\Final_Project\\Final_Project_funcs.py'>

###  Calculating SNR using signal variance and noise floor estimation

In [94]:
def calculate_snr(signal):
 
    signal_power = np.mean(np.abs(signal)**2)
    # Estimate noise floor using lower percentile of signal
    noise_floor = np.percentile(np.abs(signal)**2, 10)
    return 10 * np.log10(signal_power / noise_floor)

### Adding Gaussian noise to our signals 

In [95]:
def add_noise(signal, original_snr, target_snr_db):
    signal_power = np.mean(np.abs(signal)**2)
    target_snr = 10**(target_snr_db / 10)
    noise_power = signal_power / target_snr
    noise = np.random.normal(scale=np.sqrt(noise_power), size=signal.shape)
    return signal + noise

### Example SNR calculation with first sample

In [96]:
original_snr = calculate_snr(data_combined[0]) 

In [97]:
num_samples = data_combined.shape[0]
noisy_signals = []
snr_levels = [20, 10, 0] # SNR levels to be used as labels
for i in range(num_samples): #Loop through all available samples
    for snr_level in snr_levels:
        noisy_signal = add_noise(data_combined[i], original_snr, snr_level)
        noisy_signals.append(noisy_signal)
noisy_signals = np.array(noisy_signals)

In [98]:
#Create labels based on the number of samples and snr levels.
labels = np.repeat(np.array(range(len(snr_levels))), data_combined.shape[0]) #Example with range of snr_levels length

In [99]:
# Data Splitting and Conversion to Tensors
X_train, X_val, y_train, y_val = train_test_split(noisy_signals, labels, test_size=0.2, random_state=7)

X_train_tensor = torch.tensor(X_train, dtype=torch.float32)
X_val_tensor = torch.tensor(X_val, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train, dtype=torch.long)
y_val_tensor = torch.tensor(y_val, dtype=torch.long)

In [100]:
# Model Training with skorch

cnn = skorch.NeuralNetClassifier(
    fpf.ClayClassifierModule,
    max_epochs=10,   # chosen number of passes through training data
    lr=0.002,        # chosen learning rate for gradient descent
    optimizer=torch.optim.Adam,
    
    # Model architecture parameters
    module__n_input_channels=2,   # 2 channels for I and Q component
    module__n_output_probs=len(snr_levels), # Number of SNR classes as defined 
    module__sequence_length=2000,   # The length of each input segment

    # Training Configuration
    train_split=None,   # Using manual train/validation 
    criterion=nn.CrossEntropyLoss,   # The loss function for classification
    iterator_train__shuffle=True,
    callbacks=[('train_acc', skorch.callbacks.EpochScoring(
        scoring='accuracy',
        on_train=True,
        name='train_acc'
    ))]
)

cnn.fit(X_train_tensor, y_train_tensor)

  epoch    train_acc    train_loss       dur
-------  -----------  ------------  --------
      1       0.3418        1.0994  119.6069
      2       0.3635        1.0902  118.0479
      3       0.3786        1.0800  120.7838
      4       0.3935        1.0634  153.2753
      5       0.4130        1.0488  168.0770
      6       0.4298        1.0351  172.9604
      7       0.4351        1.0288  179.6110
      8       0.4363        1.0273  180.5807
      9       0.4363        1.0283  185.2065
     10       0.4343        1.0348  193.9336


<class 'skorch.classifier.NeuralNetClassifier'>[initialized](
  module_=ClayClassifierModule(
    (conv_network): ModuleList(
      (0): Conv1d(2, 64, kernel_size=(7,), stride=(1,))
      (1): MaxPool1d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
      (2): Conv1d(64, 128, kernel_size=(5,), stride=(1,))
      (3): Dropout(p=0.5, inplace=False)
      (4): MaxPool1d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
      (5): Conv1d(128, 256, kernel_size=(3,), stride=(1,))
      (6): Dropout(p=0.5, inplace=False)
      (7): MaxPool1d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    )
    (dense_network): ModuleList(
      (0): Linear(in_features=63232, out_features=512, bias=True)
      (1): Linear(in_features=512, out_features=256, bias=True)
    )
    (output): Linear(in_features=256, out_features=3, bias=True)
  ),
)

In [78]:
# Evaluation
val_accuracy = cnn.score(X_val_tensor, y_val_tensor)
print(f"Validation Accuracy: {val_accuracy * 100:.2f}%")

Validation Accuracy: 65.33%
